# Fine-tune GPT-2 Vietnamese for Math Word Problems

**Goal:** fine-tune `NlpHUST/gpt2-vietnamese` to solve Vietnamese math word problems.

**Inputs:** `train.json`, `valid.json`, and the local GPT-2 model folder from Kaggle Input.

**Pipeline:** load data → SFT training → generate validation outputs → evaluate by relative error.

**Rules:** Internet OFF, no extra data/API/LLM, total runtime ≤ 3 hours.


In [7]:
import os, sys, json, math, time, re, random, hashlib, inspect
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List

import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))


Torch: 2.10.0+cu128
CUDA: True | GPU count: 2
0 Tesla T4
1 Tesla T4


In [8]:
# ============================================================
# 1. Config
# ============================================================
def first_existing(*paths) -> Path:
    for p in map(Path, paths):
        if p.exists():
            return p
    raise FileNotFoundError("Không tìm thấy path nào: " + " | ".join(map(str, paths)))

DATA_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/dataset-math",
    "/kaggle/input/dataset-math",
)
MODEL_NAME = str(first_existing(
    "/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/gpt2-vietnamese",
))

TRAIN_FILE = DATA_DIR / "train.json"
VALID_FILE = DATA_DIR / "valid.json"

PROMPT_TEMPLATE = "Câu hỏi: {q}\nLời giải: "
SAFE_EOS_ID = 50256

OUTPUT_DIR = Path("/kaggle/working/gpt2_math_ckpt")
VALID_OUTPUT_PATH = Path("/kaggle/working/valid_output.json")
VALID_REPORT_PATH = Path("/kaggle/working/valid_report.json")
BASELINE_OUTPUT_PATH = Path("/kaggle/working/baseline_valid_output.json")
BASELINE_REPORT_PATH = Path("/kaggle/working/baseline_valid_report.json")

MAX_TRAIN_SAMPLES = None
MAX_VALID_SAMPLES = None

EPOCHS = 1
MAX_LENGTH = 512
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM = 8
LR = 5e-5
WARMUP_RATIO = 0.03
SEED = 42

MAX_NEW_TOKENS = 256
RUN_BASELINE_FIRST = False

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

print("TRAIN_FILE:", TRAIN_FILE)
print("VALID_FILE:", VALID_FILE)
print("MODEL_NAME:", MODEL_NAME)


TRAIN_FILE: /kaggle/input/datasets/kimanh2002/dataset-math/train.json
VALID_FILE: /kaggle/input/datasets/kimanh2002/dataset-math/valid.json
MODEL_NAME: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese


In [9]:
# Quick check: model must load offline
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, local_files_only=True)

tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID
model.config.pad_token_id = SAFE_EOS_ID
model.config.eos_token_id = SAFE_EOS_ID

print(type(tokenizer))
print(type(model))
print("vocab_size:", model.config.vocab_size)

del model
torch.cuda.empty_cache()


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<class 'transformers.models.gpt2.tokenization_gpt2.GPT2Tokenizer'>
<class 'transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel'>
vocab_size: 50257


In [10]:
# ============================================================
# 2. Data loading
# ============================================================
def load_records(path: str | Path) -> list:
    p = Path(path)
    with p.open("r", encoding="utf-8") as f:
        head = f.read(1)
        f.seek(0)
        return json.load(f) if head == "[" else [json.loads(line) for line in f if line.strip()]

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_dir(dir_path: Path, suffixes=(".bin", ".safetensors", ".json", ".txt", ".model")) -> str:
    h = hashlib.sha256()
    for p in sorted(x for x in dir_path.rglob("*") if x.is_file() and x.suffix in suffixes):
        h.update(p.relative_to(dir_path).as_posix().encode() + b"\0")
        h.update(sha256_file(p).encode() + b"\0")
    return h.hexdigest()

train_records = load_records(TRAIN_FILE)
valid_records = load_records(VALID_FILE)

if MAX_TRAIN_SAMPLES:
    train_records = train_records[:MAX_TRAIN_SAMPLES]
if MAX_VALID_SAMPLES:
    valid_records = valid_records[:MAX_VALID_SAMPLES]

print("train:", len(train_records), "| valid:", len(valid_records))
print("sample query:", train_records[0]["query_vi"][:200])


train: 100000 | valid: 1000
sample query: Bridgette và Alex sắp kết hôn. Bridgette đang mời 84 khách và Alex đang mời 2/3 số khách đó. Họ thuê một người phục vụ ăn uống để chuẩn bị bữa ăn cho từng vị khách trong tiệc cưới. Người cung cấp thực


In [11]:
# ============================================================
# 3. SFT dataset and collator
# ============================================================
class SFTDataset(Dataset):
    """Tokenize (prompt, response), mask loss on prompt + padding."""

    def __init__(self, records, tokenizer, max_length: int):
        self.records = records
        self.tok = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, i: int) -> Dict[str, List[int]]:
        rec = self.records[i]
        prompt = PROMPT_TEMPLATE.format(q=rec["query_vi"].strip())
        response = rec["response_vi"].strip()

        p_ids = self.tok(prompt, add_special_tokens=False)["input_ids"]
        r_ids = self.tok(response, add_special_tokens=False)["input_ids"] + [SAFE_EOS_ID]

        ids = (p_ids + r_ids)[: self.max_length]
        labels = ([-100] * len(p_ids) + r_ids)[: self.max_length]

        # Defensive clamp: never feed out-of-range ids to embedding layer.
        ids = [min(t, SAFE_EOS_ID) for t in ids]
        labels = [(-100 if t == -100 else min(t, SAFE_EOS_ID)) for t in labels]

        return {
            "input_ids": ids,
            "labels": labels,
            "attention_mask": [1] * len(ids),
        }

@dataclass
class PadCollator:
    pad_id: int = SAFE_EOS_ID

    def __call__(self, batch):
        maxlen = max(len(x["input_ids"]) for x in batch)
        out = {"input_ids": [], "attention_mask": [], "labels": []}
        for x in batch:
            n = len(x["input_ids"])
            pad = maxlen - n
            out["input_ids"].append(x["input_ids"] + [self.pad_id] * pad)
            out["attention_mask"].append(x["attention_mask"] + [0] * pad)
            out["labels"].append(x["labels"] + [-100] * pad)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}


In [21]:
# ============================================================
# 4. Evaluation utilities
# ============================================================
VI_ANCHORS = [
    r"Câu trả lời là\s*[:：]?",
    r"Đáp án là\s*[:：]?",
    r"Đáp án\s*[:：]",
]

EN_ANCHORS = [
    r"The answer is\s*[:：]?",
    r"####",
]

BOXED_RE = re.compile(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")
SAFE_NS = {"sqrt": math.sqrt, "pi": math.pi}


def extract_answer(text: str | None, anchors: list[str]) -> str | None:
    """Extract final answer by anchors. First anchor wins; fallback to last \\boxed{}."""
    if not text:
        return None

    for anc in anchors:
        m = re.search(anc, text)
        if m:
            tail = text[m.end():].strip()
            tail = tail.split("\n")[0]
            return tail.strip().rstrip(".。、,")

    boxes = BOXED_RE.findall(text)
    if boxes:
        return boxes[-1].strip()

    return None


def extract_gold(rec: dict) -> str | None:
    """Gold answer is read only from gold records, not from prediction files."""
    return extract_answer(rec.get("response_vi"), VI_ANCHORS)


def extract_pred(rec: dict) -> str | None:
    """Predicted answer is read only from model_output."""
    return extract_answer(rec.get("model_output"), VI_ANCHORS + EN_ANCHORS)


def parse_number(s: str | None) -> float | None:
    """Best-effort parser for finite scalar numeric answers."""
    if s is None:
        return None

    t = s.strip()
    if not t:
        return None

    # Pure Vietnamese decimal: 1,5 -> 1.5
    if re.fullmatch(r"-?\d+,\d+", t):
        try:
            return float(t.replace(",", "."))
        except ValueError:
            return None

    # Pure English decimal / integer
    if re.fullmatch(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?", t):
        try:
            val = float(t)
            return val if math.isfinite(val) else None
        except ValueError:
            return None

    # Strip variable assignment: x = 5 -> 5
    m = re.match(r"^[A-Za-z_]\w*\s*=\s*(.+)$", t)
    if m:
        t = m.group(1).strip()

    # Reject tuples / intervals early
    if t.startswith("(") and t.endswith(")") and re.search(r"\d\s*,\s*\d", t):
        return None
    if t.startswith("[") and t.endswith("]"):
        return None

    # Strip wrappers
    for _ in range(3):
        new = re.sub(r"\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}", r"(\1)", t)
        if new == t:
            break
        t = new

    t = re.sub(r"\\text\{[^}]*\}", "", t)
    t = re.sub(r"\\mathrm\{[^}]*\}", "", t)
    t = t.replace("$", "")

    for token in ("\\,", "\\!", "\\;", "\\ ", "\\left", "\\right"):
        t = t.replace(token, "")

    for token in ("\\cdot", "\\times"):
        t = t.replace(token, "*")

    # LaTeX fractions / roots
    t = re.sub(r"\\(?:d|t)?frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"((\1)/(\2))", t)
    t = re.sub(r"\\sqrt\s*\{([^{}]+)\}", r"sqrt(\1)", t)
    t = re.sub(r"\\sqrt\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", t)
    t = t.replace("\\pi", "pi")

    # Implicit multiplication
    t = re.sub(r"(\d)\s*(sqrt|pi|\()", r"\1*\2", t)
    t = re.sub(r"(\))\s*(sqrt|pi|\d)", r"\1*\2", t)
    t = re.sub(r"(pi)\s*(sqrt|pi|\d|\()", r"\1*\2", t)

    # Comma handling
    has_period = "." in t
    n_commas = t.count(",")

    if n_commas == 1 and not has_period and re.search(r"\d,\d", t):
        # Vietnamese decimal
        t = re.sub(r"(?<=\d),(?=\d)", ".", t)
    elif n_commas >= 1:
        # English thousands separator: 1,000 -> 1000
        t = re.sub(r"(?<=\d),(?=\d{3}\b)", "", t)

    t = re.sub(r"\s+", "", t)

    if not t:
        return None

    # Reject remaining tuples/lists
    if "," in t:
        return None

    # Only allow safe expression chars
    leftover = re.sub(r"sqrt|pi|\d|\.|\+|\-|\*|/|\(|\)|\^|e|E", "", t)
    if leftover:
        return None

    t = t.replace("^", "**")

    try:
        val = eval(t, {"__builtins__": {}}, SAFE_NS)
    except Exception:
        return None

    if isinstance(val, bool):
        return None

    if isinstance(val, (int, float)):
        val = float(val)
        return val if math.isfinite(val) else None

    return None


def rel_error(pred: float | None, gold: float | None) -> float | None:
    if pred is None or gold is None:
        return None

    denom = max(1.0, abs(gold))
    return abs(pred - gold) / denom


def score_one(re_val: float | None, extractable: bool) -> int:
    if not extractable:
        return 0
    if re_val is None:
        return 0
    if re_val <= 0.01:
        return 10
    if re_val <= 0.10:
        return 5
    if re_val <= 0.50:
        return 1
    return 0


def align_predictions_with_gold(pred_items: list[dict], gold_items: list[dict]) -> list[tuple[dict, dict]]:
    """
    Align predictions and gold records.

    If both sides have id, align by id.
    Otherwise, require the same length and align by order.
    """
    pred_has_id = all("id" in x for x in pred_items)
    gold_has_id = all("id" in x for x in gold_items)

    if pred_has_id and gold_has_id:
        pred_map = {str(x["id"]): x for x in pred_items}
        pairs = []

        missing = []
        for g in gold_items:
            gid = str(g["id"])
            if gid not in pred_map:
                missing.append(gid)
            else:
                pairs.append((pred_map[gid], g))

        if missing:
            raise ValueError(f"Prediction thiếu {len(missing)} id, ví dụ: {missing[:5]}")

        return pairs

    if len(pred_items) != len(gold_items):
        raise ValueError(
            f"Số lượng prediction ({len(pred_items)}) khác số lượng gold ({len(gold_items)})."
        )

    return list(zip(pred_items, gold_items))


def evaluate(pred_items: list[dict], gold_items: list[dict]) -> dict:
    pairs = align_predictions_with_gold(pred_items, gold_items)

    rows = []
    total = 0
    bucket10 = bucket5 = bucket1 = bucket0 = 0
    extractable = 0
    numeric_pairs = 0
    rel_errors = []

    for pred_rec, gold_rec in pairs:
        gold_ans = extract_gold(gold_rec)
        pred_ans = extract_pred(pred_rec)

        is_extractable = pred_ans is not None
        extractable += int(is_extractable)

        gold_num = parse_number(gold_ans)
        pred_num = parse_number(pred_ans)
        re_val = rel_error(pred_num, gold_num)

        if gold_num is not None and pred_num is not None and re_val is not None:
            numeric_pairs += 1
            rel_errors.append(re_val)

        s = score_one(re_val, is_extractable)
        total += s

        bucket10 += int(s == 10)
        bucket5 += int(s == 5)
        bucket1 += int(s == 1)
        bucket0 += int(s == 0)

        rows.append({
            "id": gold_rec.get("id", pred_rec.get("id")),
            "type": gold_rec.get("type") or pred_rec.get("type"),
            "gold_answer": gold_ans,
            "pred_answer": pred_ans,
            "gold_num": gold_num,
            "pred_num": pred_num,
            "rel_error": re_val,
            "extractable": is_extractable,
            "score": s,
        })

    n = len(rows)
    max_score = n * 10
    score_10 = total / n if n else 0.0

    return {
        "summary": {
            "n": n,
            "raw_score": total,
            "max_raw_score": max_score,
            "score_10": score_10,
            "score_pct": total / max_score if max_score else 0.0,
            "extractable": extractable,
            "numeric_pairs": numeric_pairs,
            "buckets": {
                "10": bucket10,
                "5": bucket5,
                "1": bucket1,
                "0": bucket0,
            },
            "rel_error_mean": sum(rel_errors) / len(rel_errors) if rel_errors else None,
        },
        "rows": rows,
    }


def save_evaluation_report(
    pred_path: str | Path,
    gold_records: list[dict],
    report_path: str | Path,
) -> dict:
    pred_path = Path(pred_path)
    report_path = Path(report_path)

    with pred_path.open("r", encoding="utf-8") as f:
        pred_items = json.load(f)

    result = evaluate(pred_items, gold_records)

    print(json.dumps(result["summary"], ensure_ascii=False, indent=2))

    with report_path.open("w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    print(f"Wrote {report_path}")
    return result

In [13]:
# ============================================================
# 5. Greedy generation
# ============================================================
def build_prompt(rec: dict) -> str:
    return PROMPT_TEMPLATE.format(q=rec["query_vi"].strip())

def generate_outputs(model_path_or_name: str | Path, records: list, output_path: str | Path, max_new_tokens: int = 256):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading {model_path_or_name} on {device} ...", flush=True)

    tokenizer = AutoTokenizer.from_pretrained(model_path_or_name, local_files_only=True)
    tokenizer.pad_token_id = SAFE_EOS_ID
    tokenizer.eos_token_id = SAFE_EOS_ID

    dtype = torch.float16 if device == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_path_or_name,
        torch_dtype=dtype,
        local_files_only=True,
    ).to(device)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    model.eval()

    vocab_n = model.transformer.wte.num_embeddings
    outputs, t0 = [], time.time()

    with torch.inference_mode():
        for rec in tqdm(records, desc="Generating"):
            prompt = build_prompt(rec)
            enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)

            ids = enc["input_ids"].clamp(max=vocab_n - 1)
            gen = model.generate(
                input_ids=ids,
                attention_mask=enc.get("attention_mask"),
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1,
                pad_token_id=SAFE_EOS_ID,
                eos_token_id=SAFE_EOS_ID,
            )

            text = tokenizer.decode(gen[0, ids.shape[1]:], skip_special_tokens=True)
            outputs.append({
                "id": len(outputs),
                "query_vi": rec["query_vi"],
                "type": rec.get("type"),
                "model_output": text,
            })

    output_path = Path(output_path)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(outputs, f, ensure_ascii=False, indent=2)

    out_hash = sha256_file(output_path)
    Path(str(output_path) + ".sha256.txt").write_text(out_hash + "\n", encoding="utf-8")

    dt = time.time() - t0
    print(f"Wrote {output_path} | {dt/60:.2f} min | SHA256: {out_hash}")
    return outputs


In [14]:
# ============================================================
# 6. Optional baseline
# ============================================================
if RUN_BASELINE_FIRST:
    _ = generate_outputs(MODEL_NAME, valid_records, BASELINE_OUTPUT_PATH, MAX_NEW_TOKENS)
    baseline_result = save_evaluation_report(BASELINE_OUTPUT_PATH, valid_records, BASELINE_REPORT_PATH)
else:
    print("Skip baseline.")


Skip baseline.


In [15]:
# ============================================================
# 7. Train
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    local_files_only=True
)
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    local_files_only=True
)
model.config.pad_token_id = SAFE_EOS_ID
model.config.eos_token_id = SAFE_EOS_ID


model.gradient_checkpointing_enable()
model.config.use_cache = False

train_ds = SFTDataset(train_records, tokenizer, MAX_LENGTH)
valid_ds = SFTDataset(valid_records, tokenizer, MAX_LENGTH)
collator = PadCollator(pad_id=SAFE_EOS_ID)

eff_batch = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count())
steps_per_epoch = math.ceil(len(train_ds) / eff_batch)
print(
    f"per_device_bs={PER_DEVICE_BATCH_SIZE} | grad_accum={GRAD_ACCUM} | "
    f"gpus={torch.cuda.device_count()} | effective_batch={eff_batch} | "
    f"steps/epoch={steps_per_epoch}"
)


ta_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    weight_decay=0.0,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

sig = inspect.signature(TrainingArguments.__init__)
if "eval_strategy" in sig.parameters:
    ta_kwargs["eval_strategy"] = "epoch"
else:
    ta_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**ta_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=collator,
)

t0 = time.time()
trainer.train()
train_dt = time.time() - t0
print(f"\n[train] wall time: {train_dt:.1f}s ({train_dt/60:.2f} min)")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

model_hash = sha256_dir(OUTPUT_DIR)
with (OUTPUT_DIR / "model_hash.txt").open("w", encoding="utf-8") as f:
    f.write(model_hash + "\n")

print("Saved checkpoint to:", OUTPUT_DIR)
print("Model SHA256:", model_hash)

# Free training objects before inference reload.
del trainer, model
torch.cuda.empty_cache()


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


per_device_bs=4 | grad_accum=8 | gpus=2 | effective_batch=64 | steps/epoch=1563


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,1.207019,1.188177


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


[train] wall time: 8010.8s (133.51 min)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved checkpoint to: /kaggle/working/gpt2_math_ckpt
Model SHA256: 3f1011a9db56c499d3268216385f20e2a282a343732a7c8b6bc8121ba8bfbe1b


In [16]:
# ============================================================
# 8. Inference on validation set
# ============================================================
valid_outputs = generate_outputs(OUTPUT_DIR, valid_records, VALID_OUTPUT_PATH, max_new_tokens=MAX_NEW_TOKENS)

print("\nExample output:")
print(json.dumps(valid_outputs[0], ensure_ascii=False, indent=2)[:2000])


Loading /kaggle/working/gpt2_math_ckpt on cuda ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Generating:   0%|          | 0/1000 [00:00<?, ?it/s]

Wrote /kaggle/working/valid_output.json | 27.32 min | SHA256: 17ab00e023e533cacf10d0373bdc08a69596c63b1ac8b27557b752251010e7a1

Example output:
{
  "id": 0,
  "query_vi": "Nếu Susan đang chơi một trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước tám ô, ở lượt thứ hai, cô ấy di chuyển hai ô nhưng bị đẩy lùi lại năm ô và ở lượt thứ ba. đến lượt cô ấy tiến về phía trước sáu ô, cô ấy cần di chuyển thêm bao nhiêu ô nữa để đến ô cuối và giành chiến thắng trong trò chơi?",
  "type": "GSM_Rephrased",
  "model_output": "Susan đang chơi một trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy di chuyển hai ô nên cô ấy di chuyển 2 ô. Ở lượt thứ hai, cô ấy di chuyển 2 ô nên cô ấy di chuyển 2 ô. Ở lượt thứ ba, cô ấy di chuyển 2 ô nên cô ấy di chuyển 2 ô. Tổng cộng, Susan đã di chuyển 48 ô. ####48 Đáp án là: 48"
}


In [17]:

# ============================================================
# 9. Evaluate
# ============================================================
valid_result = save_evaluation_report(
    VALID_OUTPUT_PATH,
    valid_records,
    VALID_REPORT_PATH,
)

summary = valid_result["summary"]

print("\nFinal validation score:")
print(f'{summary["raw_score"]} / {summary["max_raw_score"]}  ({summary["score_pct"]*100:.2f}%)')
print(f'Score /10: {summary["score_10"]:.2f}')
print("Buckets:", summary["buckets"])

{
  "n": 1000,
  "total_score": 344,
  "max_score": 10000,
  "score_pct": 0.0344,
  "extractable": 518,
  "numeric_pairs": 500,
  "buckets": {
    "10": 16,
    "5": 9,
    "1": 139,
    "0": 836
  },
  "rel_error_mean": 0.6511882137027881
}
Wrote /kaggle/working/valid_report.json

Final validation score:
344 / 10000  (3.44%)
Buckets: {'10': 16, '5': 9, '1': 139, '0': 836}


In [18]:
# ============================================================
# 10. Error analysis preview
# ============================================================
rows = valid_result["rows"]
bad = [(i, r) for i, r in enumerate(rows) if r.get("score", 0) == 0]
good = [(i, r) for i, r in enumerate(rows) if r.get("score", 0) == 10]

print("Good:", len(good), "| Bad:", len(bad))

for i, r in bad[:2]:
    pred = valid_outputs[i]
    gold = valid_records[i]
    print("=" * 100)
    print("IDX:", i, "| type:", gold.get("type"), "| rel_error:", r.get("rel_error"))
    print("QUERY:", gold["query_vi"][:500])
    print("\nGOLD:", gold["response_vi"][-500:])
    print("\nPRED:", pred["model_output"][:1000])


Good: 16 | Bad: 836
IDX: 1 | type: MATH_Rephrased | rel_error: None
QUERY: Nếu $\angle PQR = \angle PRQ$, và độ dài của QR và PR lần lượt là 5 và 7 thì chu vi của tam giác PQR là bao nhiêu?

GOLD: Vì $\angle PQR = \angle PRQ$ nên ta biết tam giác PQR là tam giác cân. Điều này có nghĩa là PQ cũng có độ dài bằng 7. Khi đó chu vi của tam giác PQR là $PQ + QR + PR = 7 + 5 + 7 = \boxed{19}$.Câu trả lời là: 19

PRED: Chúng ta có thể viết lại phương trình dưới dạng $\angle PQR = \angle PRQ$, trong đó $P$ là độ dài của QR và PR lần lượt là 5 và 7. Để tìm chu vi của tam giác PQR, chúng ta có thể sử dụng công thức $\angle PRQ = \angle PRQ$, trong đó $P$ là độ dài của QR và PR lần lượt là 5 và 7. Chúng ta có thể thay thế các giá trị này vào phương trình: $\angle PRQ = \angle PRQ$ và $\angle PRQ = \angle PRQ$ Để tìm chu vi của tam giác PQR, chúng ta có thể thay thế các giá trị này vào phương trình: $\angle PRQ = \angle PRQ$ và $\angle PRQ = \angle PRQ$ Để tìm chu vi của tam giác PQR, chúng ta có t

In [19]:
# ============================================================
# 11. List saved artifacts
# ============================================================
for p in [
    VALID_OUTPUT_PATH,
    VALID_REPORT_PATH,
    Path(str(VALID_OUTPUT_PATH) + ".sha256.txt"),
    OUTPUT_DIR / "model_hash.txt",
]:
    print(p, "exists=", p.exists(), "size=", p.stat().st_size if p.exists() else None)

print("\nKaggle output folder:")
for p in sorted(Path("/kaggle/working").glob("*")):
    print("-", p)


/kaggle/working/valid_output.json exists= True size= 1066077
/kaggle/working/valid_report.json exists= True size= 218681
/kaggle/working/valid_output.json.sha256.txt exists= True size= 65
/kaggle/working/gpt2_math_ckpt/model_hash.txt exists= True size= 65

Kaggle output folder:
- /kaggle/working/.virtual_documents
- /kaggle/working/gpt2_math_ckpt
- /kaggle/working/valid_output.json
- /kaggle/working/valid_output.json.sha256.txt
- /kaggle/working/valid_report.json
